# Update All Site CSVs

Runs all four data-update steps in order so a single notebook run fully refreshes
`sombrero_preds.csv`, `sk_preds.csv`, and `molasses_preds.csv`.

| Section | Variables updated | Source |
|---------|------------------|--------|
| 1 · CRW | `meanSst`, `meanSsta`, `meanHs`, `meanDhw` | NOAA Coral Reef Watch daily virtual-station files |
| 2 · NCEP | `airTemp`, `airTempInter`, `zonalWind`, `zonalWindInter` | NOAA PSL NCEP Reanalysis via OPeNDAP |
| 3 · ONI | `oni`, `oniInter` | NOAA PSL nina34.anom.data (downloaded fresh) |
| 4 · OISST | `loop_curr_sst` | NOAA OISST weekly SST via OPeNDAP |

**Columns NOT updated:** `sdSst`, `sdSsta`, `sdDhw`, `sdHs`, `bleachClass`, `date`,
`geoHeight`, `geoHeightInter`, `meridWind`, `meridWindInter`
(no source available or not used in predictions)


In [17]:
import sys
import urllib.request
from io import StringIO
import numpy as np
import pandas as pd
import xarray as xr
import fsspec
import warnings
from pathlib import Path
from datetime import date

warnings.filterwarnings('ignore')
print(f'xarray  {xr.__version__}')
print(f'pandas  {pd.__version__}')


xarray  2024.7.0
pandas  2.3.1


## Configuration

In [18]:
# ── CRW virtual-station URLs ──────────────────────────────────────────────────
CRW_URLS = {
    'sombrero': 'https://coralreefwatch.noaa.gov/product/vs_single_pixel_exp/data/'
                'AtlanticOcean_Florida_FloridaKeys_SombreroReef.txt',
    'sk':       'https://coralreefwatch.noaa.gov/product/vs_single_pixel_exp/data/'
                'AtlanticOcean_Florida_FloridaKeys_SandKey.txt',
    'molasses': 'https://coralreefwatch.noaa.gov/product/vs_single_pixel_exp/data/'
                'AtlanticOcean_Florida_FloridaKeys_MolassesReef.txt',
}

# ── NCEP OPeNDAP ──────────────────────────────────────────────────────────────
BASE_OPENDAP  = 'https://psl.noaa.gov/thredds/dodsC/Datasets/ncep'
URL_AIRTEMP   = BASE_OPENDAP + '/air.sfc.{year}.nc'
URL_UWND      = BASE_OPENDAP + '/uwnd.{year}.nc'
TARGET_LAT    = 25.0    # °N  (nearest to Florida Keys)
TARGET_LON    = 280.0   # °E  (= 80°W)
UWND_LEVEL    = 925     # mb

# ── ONI ───────────────────────────────────────────────────────────────────────
ONI_URL     = 'https://psl.noaa.gov/data/correlation/nina34.anom.data'
ONI_MISSING = -99.0     # values below this threshold are missing

# ── NOAA OISST (loop current SST) ─────────────────────────────────────────────
NOAA_OPENDAP = ('https://psl.noaa.gov/thredds/dodsC/Datasets/'
                'noaa.oisst.v2.highres/sst.week.mean.nc')
NOAA_HTTP    = ('https://downloads.psl.noaa.gov/Datasets/'
                'noaa.oisst.v2.highres/sst.week.mean.nc')
LAT_MIN, LAT_MAX = 21, 28
LON_MIN, LON_MAX = 270, 281
CLIMO_START = '1991-01-01'
CLIMO_END   = '2020-12-31'
INDEX_START = '1985-01-01'

# ── NCEP download years ───────────────────────────────────────────────────────
CURRENT_YEAR   = 2026
DOWNLOAD_YEARS = list(range(1985, CURRENT_YEAR + 1))

# ── CSV paths ─────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../Data')
SITE_FILES = {
    'sombrero': DATA_DIR / 'sombrero_preds.csv',
    'sk':       DATA_DIR / 'sk_preds.csv',
    'molasses': DATA_DIR / 'molasses_preds.csv',
}
SITES = list(SITE_FILES.keys())

print('Configuration loaded.')
print(f'NCEP: {len(DOWNLOAD_YEARS)} years ({DOWNLOAD_YEARS[0]}–{DOWNLOAD_YEARS[-1]})')
print(f'Sites: {SITES}')


Configuration loaded.
NCEP: 42 years (1985–2026)
Sites: ['sombrero', 'sk', 'molasses']


---
## Section 1 — CRW Data

Downloads the NOAA Coral Reef Watch daily virtual-station file for each reef site,
aggregates daily SST/SSTA/HS/DHW to ISO weekly means, overwrites those columns in
the site CSVs, and appends any new (year, week) rows not yet in the CSV.

**Columns updated:** `meanSst`, `meanSsta`, `meanHs`, `meanDhw`


In [19]:
def fetch_crw_weekly(url, site_name):
    '''Download CRW daily text file, return weekly-aggregated DataFrame.'''
    with urllib.request.urlopen(url) as resp:
        text = resp.read().decode('utf-8')
    lines = text.splitlines()

    # Find the column-header line (starts with YYYY)
    header_idx = next(
        (i for i, ln in enumerate(lines) if ln.strip().startswith('YYYY')),
        None,
    )
    if header_idx is None:
        raise ValueError(f'YYYY header not found in {url}')

    data_text = '\n'.join(lines[header_idx:])
    df = pd.read_csv(
        StringIO(data_text),
        skipinitialspace=True,
        names=['year', 'month', 'day', 'sst', 'ssta', 'hs', 'dhw', 'baa', 'baa7'],
        skiprows=1,
    )
    df = df[['year', 'month', 'day', 'sst', 'ssta', 'hs', 'dhw']].copy()

    # Assign ISO year/week
    dates = pd.to_datetime(dict(year=df['year'], month=df['month'], day=df['day']))
    iso   = dates.dt.isocalendar()
    df['iso_year'] = iso.year.values.astype(int)
    df['iso_week'] = iso.week.values.astype(int)

    # Weekly mean
    weekly = (
        df.groupby(['iso_year', 'iso_week'])[['sst', 'ssta', 'hs', 'dhw']]
        .mean()
        .reset_index()
    )
    weekly.columns = ['iso_year', 'iso_week', 'meanSst', 'meanSsta', 'meanHs', 'meanDhw']

    last_y = int(weekly['iso_year'].max())
    last_w = int(weekly.loc[weekly['iso_year'] == last_y, 'iso_week'].max())
    print(f'  {site_name}: {len(weekly)} weekly rows  '
          f'({int(weekly["iso_year"].min())}w{int(weekly["iso_week"].min()):02d}'
          f' → {last_y}w{last_w:02d})')
    return weekly


In [20]:
print('Downloading CRW data...')
crw_weekly = {}
for site, url in CRW_URLS.items():
    crw_weekly[site] = fetch_crw_weekly(url, site)
print('Done.')


  sombrero: 2166 weekly rows  (1985w01 → 2026w27)
  sk: 2166 weekly rows  (1985w01 → 2026w27)
  molasses: 2166 weekly rows  (1985w01 → 2026w27)
Done.


In [21]:
print('Updating CSVs with CRW data...')
for site, csv_path in SITE_FILES.items():
    df   = pd.read_csv(csv_path)
    crw  = crw_weekly[site]
    n_orig = len(df)

    # ── Overwrite existing rows ───────────────────────────────────────────────
    df = df.merge(
        crw.rename(columns={
            'iso_year': '_iy', 'iso_week': '_iw',
            'meanSst': '_sst', 'meanSsta': '_ssta',
            'meanHs': '_hs', 'meanDhw': '_dhw',
        }),
        left_on=['year', 'week'], right_on=['_iy', '_iw'],
        how='left',
    ).drop(columns=['_iy', '_iw'])

    mask = df['_sst'].notna()
    for tmp, col in [('_sst', 'meanSst'), ('_ssta', 'meanSsta'),
                     ('_hs', 'meanHs'), ('_dhw', 'meanDhw')]:
        df.loc[mask, col] = df.loc[mask, tmp]
    df = df.drop(columns=['_sst', '_ssta', '_hs', '_dhw'])

    # ── Append new rows ───────────────────────────────────────────────────────
    existing_yw = set(zip(df['year'].astype(int), df['week'].astype(int)))
    new_crw = crw[
        ~crw.apply(
            lambda r: (int(r['iso_year']), int(r['iso_week'])) in existing_yw,
            axis=1,
        )
    ].sort_values(['iso_year', 'iso_week'])

    n_new = len(new_crw)
    if n_new > 0:
        last_tw = int(df['totalWeek'].max())
        new_rows = []
        for _, r in new_crw.iterrows():
            last_tw += 1
            row = {col: np.nan for col in df.columns}
            row.update({
                'year':     int(r['iso_year']),
                'week':     int(r['iso_week']),
                'site':     site,
                'totalWeek': last_tw,
                'meanSst':   r['meanSst'],
                'meanSsta':  r['meanSsta'],
                'meanHs':    r['meanHs'],
                'meanDhw':   r['meanDhw'],
            })
            new_rows.append(row)
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    df.to_csv(csv_path, index=False)
    print(f'  {site}: {int(mask.sum())} rows overwritten, {n_new} new rows added '
          f'→ {len(df)} total rows')

print('CRW update complete.')


Updating CSVs with CRW data...
  sombrero: 2166 rows overwritten, 0 new rows added → 2200 total rows
  sk: 2166 rows overwritten, 0 new rows added → 2200 total rows
  molasses: 2166 rows overwritten, 0 new rows added → 2200 total rows
CRW update complete.


In [22]:
# Quick verification
for site, csv_path in SITE_FILES.items():
    df = pd.read_csv(csv_path)
    print(f'\n{site} — last 5 rows:')
    print(df[['year', 'week', 'totalWeek', 'meanSst', 'meanSsta',
              'meanHs', 'meanDhw']].tail(5).to_string(index=False))



sombrero — last 5 rows:
 year  week  totalWeek   meanSst  meanSsta   meanHs  meanDhw
 2026    23       2196 29.455714  1.619257 0.098571 0.000000
 2026    24       2197 29.041429  0.782714 0.000000 0.000000
 2026    25       2198 30.420000  1.826000 0.710000 0.071229
 2026    26       2199 31.087143  2.273814 1.377143 1.113471
 2026    27       2200 31.375714  2.343043 1.665714 2.589386

sk — last 5 rows:
 year  week  totalWeek   meanSst  meanSsta   meanHs  meanDhw
 2026    23       2196 29.364286  1.687829 0.084286 0.000000
 2026    24       2197 29.131429  1.032714 0.010000 0.000000
 2026    25       2198 30.351429  1.913429 0.711429 0.067343
 2026    26       2199 31.035714  2.369043 1.395714 1.151229
 2026    27       2200 31.237143  2.341814 1.597143 2.561629

molasses — last 5 rows:
 year  week  totalWeek   meanSst  meanSsta   meanHs  meanDhw
 2026    23       2196 29.044286  1.323957 0.000000 0.000000
 2026    24       2197 28.968571  0.866629 0.000000 0.000000
 2026    25     

---
## Section 2 — NCEP Air Temp + Zonal Wind

Downloads NCEP Reanalysis surface air temperature and 925 mb zonal wind via
OPeNDAP for all years 1985–present, aggregates daily to ISO weekly means,
and overwrites those columns in all three site CSVs.

**Columns updated:** `airTemp`, `airTempInter`, `zonalWind`, `zonalWindInter`
**Note:** `airTempInter` is in **°C** (converted from Kelvin during download).


In [23]:
def load_point_lazy(url, var, lat, lon, level=None):
    '''Open NCEP yearly file via OPeNDAP, select nearest grid point, load 1-D series.'''
    ds = xr.open_dataset(url, engine='netcdf4', decode_times=False)
    valid_mask = ds['time'].values < 1e10
    if not valid_mask.all():
        ds = ds.isel(time=valid_mask)
    ds = xr.decode_cf(ds)
    da = ds[var]
    if level is not None:
        da = da.sel(level=level, method='nearest')
    da = da.sel(lat=lat, lon=lon, method='nearest')
    da.load()
    ds.close()
    return da, float(da.lat), float(da.lon)


def daily_to_weekly(da, offset=0.0):
    '''Average daily DataArray to ISO weekly means; apply additive offset.'''
    times   = pd.DatetimeIndex(da.time.values)
    iso_cal = times.isocalendar()
    df = pd.DataFrame({
        'value':    da.values.astype(float) + offset,
        'iso_year': iso_cal.year.values.astype(int),
        'iso_week': iso_cal.week.values.astype(int),
    })
    return df.groupby(['iso_year', 'iso_week'])['value'].mean().reset_index()


print('NCEP helper functions defined.')


NCEP helper functions defined.


In [24]:
print('Downloading NCEP air temperature (surface)...')
airtemp_parts = []
for year in DOWNLOAD_YEARS:
    da, sel_lat, sel_lon = load_point_lazy(
        URL_AIRTEMP.format(year=year), 'air', TARGET_LAT, TARGET_LON
    )
    airtemp_parts.append(da)
    if year in [DOWNLOAD_YEARS[0], DOWNLOAD_YEARS[-1]]:
        print(f'  {year}: {len(da)} obs  |  cell: {sel_lat:.1f}°N {sel_lon:.1f}°E  '
              f'|  raw [{float(da.min()):.1f}, {float(da.max()):.1f}] K')
    else:
        print(f'  {year}: {len(da)} obs', end='\r')

da_airtemp = xr.concat(airtemp_parts, dim='time')
print(f'\nAir temp: {len(da_airtemp)} daily obs  '
      f'({str(da_airtemp.time.values[0])[:10]} → {str(da_airtemp.time.values[-1])[:10]})')

print('\nDownloading NCEP 925 mb zonal wind...')
uwnd_parts = []
for year in DOWNLOAD_YEARS:
    da, sel_lat, sel_lon = load_point_lazy(
        URL_UWND.format(year=year), 'uwnd', TARGET_LAT, TARGET_LON, level=UWND_LEVEL
    )
    uwnd_parts.append(da)
    if year in [DOWNLOAD_YEARS[0], DOWNLOAD_YEARS[-1]]:
        print(f'  {year}: {len(da)} obs  |  cell: {sel_lat:.1f}°N {sel_lon:.1f}°E  '
              f'|  [{float(da.min()):.2f}, {float(da.max()):.2f}] m/s')
    else:
        print(f'  {year}: {len(da)} obs', end='\r')

da_uwnd = xr.concat(uwnd_parts, dim='time')
print(f'\nZonal wind: {len(da_uwnd)} daily obs  '
      f'({str(da_uwnd.time.values[0])[:10]} → {str(da_uwnd.time.values[-1])[:10]})')


  1985: 365 obs  |  cell: 25.0°N 280.0°E  |  raw [265.9, 307.5] K
  2026: 187 obs  |  cell: 25.0°N 280.0°E  |  raw [282.5, 302.0] K

Air temp: 15161 daily obs  (1985-01-01 → 2026-07-06)

  1985: 365 obs  |  cell: 25.0°N 280.0°E  |  [-14.26, 11.13] m/s
  2026: 187 obs  |  cell: 25.0°N 280.0°E  |  [-12.01, 10.58] m/s

Zonal wind: 15161 daily obs  (1985-01-01 → 2026-07-06)


In [25]:
# Aggregate to weekly
weekly_airtemp = daily_to_weekly(da_airtemp, offset=-273.15).rename(
    columns={'value': 'airTemp_new'}
)
weekly_uwnd = daily_to_weekly(da_uwnd).rename(columns={'value': 'zonalWind_new'})
print(f'Weekly rows: airTemp={len(weekly_airtemp)}, zonalWind={len(weekly_uwnd)}')

print('\nUpdating CSVs with NCEP data...')
for site, csv_path in SITE_FILES.items():
    df = pd.read_csv(csv_path)
    n_orig = len(df)

    # Air temperature
    df = df.merge(
        weekly_airtemp.rename(columns={'airTemp_new': '_at'}),
        left_on=['year', 'week'], right_on=['iso_year', 'iso_week'], how='left',
    ).drop(columns=['iso_year', 'iso_week'])
    mask_at = df['_at'].notna()
    df.loc[mask_at, 'airTemp']      = df.loc[mask_at, '_at']
    df.loc[mask_at, 'airTempInter'] = df.loc[mask_at, '_at']
    df = df.drop(columns=['_at'])

    # Zonal wind
    df = df.merge(
        weekly_uwnd.rename(columns={'zonalWind_new': '_zw'}),
        left_on=['year', 'week'], right_on=['iso_year', 'iso_week'], how='left',
    ).drop(columns=['iso_year', 'iso_week'])
    mask_zw = df['_zw'].notna()
    df.loc[mask_zw, 'zonalWind']      = df.loc[mask_zw, '_zw']
    df.loc[mask_zw, 'zonalWindInter'] = df.loc[mask_zw, '_zw']
    df = df.drop(columns=['_zw'])

    assert len(df) == n_orig, f'Row count changed for {site}!'
    df.to_csv(csv_path, index=False)
    print(f'  {site}: airTemp updated={int(mask_at.sum())} (NaN={int(df["airTemp"].isna().sum())})  '
          f'| zonalWind updated={int(mask_zw.sum())} (NaN={int(df["zonalWind"].isna().sum())})')

print('NCEP update complete.')


Weekly rows: airTemp=2167, zonalWind=2167

Updating CSVs with NCEP data...
  sombrero: airTemp updated=2166 (NaN=34)  | zonalWind updated=2166 (NaN=34)
  sk: airTemp updated=2166 (NaN=34)  | zonalWind updated=2166 (NaN=34)
  molasses: airTemp updated=2166 (NaN=34)  | zonalWind updated=2166 (NaN=34)
NCEP update complete.


---
## Section 3 — ONI (Niño 3.4 Anomaly)

Downloads `nina34.anom.data` fresh from NOAA PSL, parses the wide-format annual
table, builds monthly anchor points (anchored at the 15th of each month → ISO week),
interpolates between anchors, and extrapolates beyond the last available month using
the slope of the two most recent anchors.

**Columns updated:** `oni`, `oniInter`


In [26]:
print(f'Downloading nina34.anom.data from {ONI_URL} ...')
with urllib.request.urlopen(ONI_URL) as resp:
    oni_content = resp.read().decode('utf-8')

# Parse wide-format: each data line has year + 12 monthly values
records = []
for line in oni_content.splitlines():
    parts = line.split()
    if len(parts) == 13:
        try:
            yr = int(parts[0])
        except ValueError:
            continue
        for month_idx, val_str in enumerate(parts[1:], start=1):
            try:
                val = float(val_str)
            except ValueError:
                continue
            if val > ONI_MISSING:
                records.append({'year': yr, 'month': month_idx, 'oni': val})

oni_monthly = pd.DataFrame(records)
print(f'Valid ONI records: {len(oni_monthly)}  '
      f'({int(oni_monthly["year"].min())}-{int(oni_monthly["month"].min()):02d}'
      f' → {int(oni_monthly["year"].max())}-{int(oni_monthly["month"].max()):02d})')
print('Most recent months:')
print(oni_monthly.tail(8).to_string(index=False))


Valid ONI records: 918  (1950-01 → 2026-12)
Most recent months:
 year  month   oni
 2025     11 -0.70
 2025     12 -0.67
 2026      1 -0.58
 2026      2 -0.27
 2026      3 -0.01
 2026      4  0.29
 2026      5  0.90
 2026      6  1.44


In [27]:
def month_to_iso_yearweek(yr, mo, day=15):
    d   = date(yr, mo, day)
    iso = d.isocalendar()
    return int(iso.year), int(iso.week)

# Build anchor list
anchors = []
for _, row in oni_monthly.iterrows():
    iso_y, iso_w = month_to_iso_yearweek(int(row['year']), int(row['month']))
    anchors.append({'iso_year': iso_y, 'iso_week': iso_w, 'oni': row['oni']})

anchor_df = (
    pd.DataFrame(anchors)
    .drop_duplicates(subset=['iso_year', 'iso_week'], keep='first')
    .reset_index(drop=True)
)
print(f'Anchor points: {len(anchor_df)}')
print('Most recent:')
print(anchor_df[anchor_df['iso_year'] >= 2026].to_string(index=False))


Anchor points: 918
Most recent:
 iso_year  iso_week   oni
     2026         3 -0.58
     2026         7 -0.27
     2026        11 -0.01
     2026        16  0.29
     2026        20  0.90
     2026        25  1.44


In [28]:
print('Updating CSVs with ONI data...')
for site, csv_path in SITE_FILES.items():
    df     = pd.read_csv(csv_path)
    n_rows = len(df)
    tw_vals = df['totalWeek'].values

    yw_to_idx  = {(int(r['year']), int(r['week'])): i for i, r in df.iterrows()}
    tw_to_idx  = {int(df.at[i, 'totalWeek']): i for i in range(n_rows)}

    # Map anchors to totalWeek values within the CSV
    anchor_tw   = []
    anchor_vals = []
    for _, a in anchor_df.iterrows():
        idx = yw_to_idx.get((a['iso_year'], a['iso_week']))
        if idx is not None:
            anchor_tw.append(int(df.at[idx, 'totalWeek']))
            anchor_vals.append(a['oni'])
    anchor_tw   = np.array(anchor_tw,   dtype=float)
    anchor_vals = np.array(anchor_vals, dtype=float)

    # oni: anchor values only
    df['oni'] = np.nan
    for tw, val in zip(anchor_tw, anchor_vals):
        idx = tw_to_idx.get(int(tw))
        if idx is not None:
            df.at[idx, 'oni'] = val

    # oniInter: interpolate + extrapolate
    tw_min = anchor_tw[0]
    tw_max = anchor_tw[-1]
    extrap_slope = (anchor_vals[-1] - anchor_vals[-2]) / (anchor_tw[-1] - anchor_tw[-2])

    oniInter = np.full(n_rows, np.nan)
    mask_interior = (tw_vals >= tw_min) & (tw_vals <= tw_max)
    oniInter[mask_interior] = np.interp(tw_vals[mask_interior], anchor_tw, anchor_vals)
    mask_extrap = tw_vals > tw_max
    oniInter[mask_extrap] = (
        anchor_vals[-1] + extrap_slope * (tw_vals[mask_extrap] - tw_max)
    )
    df['oniInter'] = oniInter

    df.to_csv(csv_path, index=False)
    print(f'  {site}: oni anchors={int(df["oni"].notna().sum())}  '
          f'| oniInter non-NaN={int(df["oniInter"].notna().sum())}  '
          f'| extrap slope={extrap_slope:.4f}/week')

print('ONI update complete.')


Updating CSVs with ONI data...
  sombrero: oni anchors=498  | oniInter non-NaN=2198  | extrap slope=0.1080/week
  sk: oni anchors=498  | oniInter non-NaN=2198  | extrap slope=0.1080/week
  molasses: oni anchors=498  | oniInter non-NaN=2198  | extrap slope=0.1080/week
ONI update complete.


---
## Section 4 — Loop Current SST

Downloads the NOAA OISST weekly SST dataset via OPeNDAP (regional subset only),
computes the 1991–2020 weekly climatology, subtracts it to get anomalies, and
takes the cosine-latitude-weighted spatial mean over the Loop Current / Caribbean
region (lat 21–28°N, lon 270–281°E).

**Column updated:** `loop_curr_sst`


In [29]:
print(f'Connecting to NOAA OISST via OPeNDAP...')
try:
    ds_lazy = xr.open_dataset(NOAA_OPENDAP, engine='netcdf4')
    print('OPeNDAP connection successful.')
except Exception as e:
    print(f'OPeNDAP failed ({e}), falling back to HTTP download (~400 MB)...')
    ds_lazy = xr.open_dataset(fsspec.open(NOAA_HTTP).open())

print(f'Dataset: {str(ds_lazy.time.values[0])[:10]} → {str(ds_lazy.time.values[-1])[:10]}'
      f'  ({ds_lazy.dims["time"]} time steps)')

# Slice region BEFORE loading — only the small subset is downloaded
sst_region = ds_lazy.sst.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
print(f'Regional slice shape: {sst_region.shape}  (time × lat × lon)')
print('Loading regional data...')
sst_region.load()
print(f'Done.  Memory: ~{sst_region.nbytes/1e6:.1f} MB  '
      f'mean={float(sst_region.mean()):.2f}°C')


Connecting to NOAA OISST via OPeNDAP...
OPeNDAP connection successful.
Dataset: 1981-09-06 → 2026-06-28  (2339 time steps)
Regional slice shape: (2339, 28, 44)  (time × lat × lon)
Loading regional data...
Done.  Memory: ~11.5 MB  mean=26.89°C


In [30]:
# ── 1991–2020 climatology ─────────────────────────────────────────────────────
iso_week = sst_region['time'].dt.isocalendar().week.rename('iso_week')
sst_region = sst_region.assign_coords(iso_week=iso_week)

print('Computing 1991–2020 climatology...')
climo_period = sst_region.sel(time=slice(CLIMO_START, CLIMO_END))
climatology  = climo_period.groupby('iso_week').mean('time')
print(f'Climatology shape: {climatology.shape}')

# ── Anomalies ─────────────────────────────────────────────────────────────────
print('Computing anomalies...')
anomalies = sst_region.groupby('iso_week') - climatology

# ── Area-weighted spatial mean → loop_curr_sst index ─────────────────────────
weights   = np.cos(np.deg2rad(anomalies['lat']))
index_all = anomalies.weighted(weights).mean(('lon', 'lat'))
index     = index_all.sel(time=slice(INDEX_START, None))
print(f'Index: {str(index.time.values[0])[:10]} → {str(index.time.values[-1])[:10]}'
      f'  ({len(index)} weekly obs)')

df_noaa = pd.DataFrame({
    'noaa_date':     pd.DatetimeIndex(index.time.values),
    'loop_curr_sst': index.values.astype(float),
})
iso_cal          = df_noaa['noaa_date'].dt.isocalendar()
df_noaa['iso_year'] = iso_cal.year.values.astype(int)
df_noaa['iso_week'] = iso_cal.week.values.astype(int)
print('Index ready.')


Computing 1991–2020 climatology...
Climatology shape: (53, 28, 44)
Computing anomalies...
Index: 1985-01-06 → 2026-06-28  (2165 weekly obs)
Index ready.


In [31]:
print('Updating CSVs with loop_curr_sst...')
loop_curr_sst_infilled = {}   # track in-fill per site for the verification cell

for site, csv_path in SITE_FILES.items():
    df     = pd.read_csv(csv_path)
    n_orig = len(df)

    if 'loop_curr_sst' in df.columns:
        df = df.drop(columns=['loop_curr_sst'])

    df = df.merge(
        df_noaa[['iso_year', 'iso_week', 'loop_curr_sst']],
        left_on=['year', 'week'], right_on=['iso_year', 'iso_week'],
        how='left',
    ).drop(columns=['iso_year', 'iso_week'])

    assert len(df) == n_orig, f'Row count changed for {site}!'

    # If the latest row that has CRW data is missing loop_curr_sst (OISST not yet
    # published for that week), copy the previous week's value forward.
    infilled = False
    last_crw_idx = df['meanSst'].last_valid_index()
    if last_crw_idx is not None and pd.isna(df.at[last_crw_idx, 'loop_curr_sst']):
        pos = df.index.get_loc(last_crw_idx)
        if pos > 0:
            prev_valid = df['loop_curr_sst'].iloc[:pos].last_valid_index()
            if prev_valid is not None:
                df.at[last_crw_idx, 'loop_curr_sst'] = df.at[prev_valid, 'loop_curr_sst']
                infilled = True

    loop_curr_sst_infilled[site] = infilled

    df.to_csv(csv_path, index=False)
    n_filled = int(df['loop_curr_sst'].notna().sum())
    n_nan    = int(df['loop_curr_sst'].isna().sum())
    infill_note = ' (latest week in-filled from prior week)' if infilled else ''
    print(f'  {site}: {n_filled} filled, {n_nan} NaN → {csv_path.name}{infill_note}')

print('OISST update complete.')


Updating CSVs with loop_curr_sst...
  sombrero: 2166 filled, 34 NaN → sombrero_preds.csv (latest week in-filled from prior week)
  sk: 2166 filled, 34 NaN → sk_preds.csv (latest week in-filled from prior week)
  molasses: 2166 filled, 34 NaN → molasses_preds.csv (latest week in-filled from prior week)
OISST update complete.


---
## Final Verification

Shows the last 5 rows of each CSV and a column-completeness summary.


In [32]:
COLS_TO_CHECK = [
    'year', 'week', 'totalWeek',
    'meanSst', 'meanSsta', 'meanHs', 'meanDhw',
    'airTempInter', 'zonalWindInter',
    'oniInter', 'loop_curr_sst',
]

for site, csv_path in SITE_FILES.items():
    df = pd.read_csv(csv_path)
    print(f'\n{"="*60}')
    print(f'{site.upper()}  ({len(df)} rows  |  '
          f'{int(df["year"].min())}w{int(df["week"].min()):02d}'
          f' → {int(df["year"].max())}w{int(df["week"].max()):02d})')
    print('='*60)
    print(df[COLS_TO_CHECK].tail(5).to_string(index=False))
    print()
    print('NaN counts for updated columns:')
    for col in COLS_TO_CHECK[3:]:
        n_nan = int(df[col].isna().sum())
        status = 'OK' if n_nan == 0 else f'{n_nan} NaN'
        print(f'  {col:<22} {status}')
    if loop_curr_sst_infilled.get(site):
        print('  → SST data not available for latest week; in-filled with the previous week\'s data')



SOMBRERO  (2200 rows  |  1985w01 → 2026w53)
 year  week  totalWeek   meanSst  meanSsta   meanHs  meanDhw  airTempInter  zonalWindInter  oniInter  loop_curr_sst
 2026    23       2196 29.455714  1.619257 0.098571 0.000000     26.629999       -3.294288     1.224       0.263475
 2026    24       2197 29.041429  0.782714 0.000000 0.000000     27.474289       -4.047143     1.332       0.557260
 2026    25       2198 30.420000  1.826000 0.710000 0.071229     28.341425       -0.882858     1.440       0.723551
 2026    26       2199 31.087143  2.273814 1.377143 1.113471     28.612848       -3.661431     1.548       1.043508
 2026    27       2200 31.375714  2.343043 1.665714 2.589386     28.584279       -2.174286     1.656       1.043508

NaN counts for updated columns:
  meanSst                OK
  meanSsta               OK
  meanHs                 OK
  meanDhw                OK
  airTempInter           OK
  zonalWindInter         OK
  oniInter               2 NaN
  loop_curr_sst          34